# Faruq-v3 — Circle-CPE Matched Objective Screening (Colab v2)

Robust fix: CPE0/CPE7 checkpoints are discovered recursively from the Drive `experiments` tree instead of assuming a stale experiment path. Missing validation reports are regenerated from the existing checkpoints only; no CPE retraining. Test is never extracted/opened.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, tarfile, time, json
from pathlib import Path
REPO=Path('/content/coffee-bean-detection')
BRANCH='agent/circle-cpe-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src'))
os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())


In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(), 'Aktifkan GPU.'
BASE_REQUIRED=(
 'bundles/faruq-development-v3-grouped.tar',
 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
 'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/D0FT_seed42_val.json',
)
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=BASE_REQUIRED)
ARCHIVE=require_project_artifact(PROJECT_ROOT,BASE_REQUIRED[0])
D0_CHECKPOINT=require_project_artifact(PROJECT_ROOT,BASE_REQUIRED[1])
D0FT_REPORT=require_project_artifact(PROJECT_ROOT,BASE_REQUIRED[2])

def find_unique_checkpoint(run_name):
    experiments=PROJECT_ROOT/'experiments'
    matches=[p for p in experiments.rglob('best.pt') if p.parent.name=='weights' and p.parent.parent.name==run_name]
    if not matches:
        raise FileNotFoundError(f'Tidak menemukan {run_name}/weights/best.pt di {experiments}')
    if len(matches)>1:
        raise RuntimeError(f'Checkpoint {run_name} ambigu: {[str(p) for p in matches]}')
    return matches[0]

CPE0_CHECKPOINT=find_unique_checkpoint('CPE0_seed42')
CPE7_CHECKPOINT=find_unique_checkpoint('CPE7_seed42')
DATA_ROOT=Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY=DATA_ROOT/'faruq_grouped_summary.json'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert GROUPED_SUMMARY.is_file()
assert not (DATA_ROOT/'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT_ROOT=PROJECT_ROOT/'experiments/faruq-v3-circle-cpe-screening-v1'
CONTROL_REPORT_DIR=OUTPUT_ROOT/'control_val_reports'
CONTROL_REPORT_DIR.mkdir(parents=True,exist_ok=True)
CPE0_REPORT=CONTROL_REPORT_DIR/'CPE0_seed42_val.json'
CPE7_REPORT=CONTROL_REPORT_DIR/'CPE7_seed42_val.json'
print('GPU:',torch.cuda.get_device_name(0))
print('OUTPUT:',OUTPUT_ROOT)
print('CPE0 CHECKPOINT:',CPE0_CHECKPOINT)
print('CPE7 CHECKPOINT:',CPE7_CHECKPOINT)


In [ ]:
# Rebuild missing matched-control validation reports from existing checkpoints only.
from coffee_detector.evaluate import evaluate
if not CPE0_REPORT.is_file():
    print('Membuat ulang CPE0 val report dari checkpoint lama...')
    evaluate(CPE0_CHECKPOINT,DATA_ROOT,CPE0_REPORT,split='val',device='0')
if not CPE7_REPORT.is_file():
    print('Membuat ulang CPE7 val report dari checkpoint lama...')
    evaluate(CPE7_CHECKPOINT,DATA_ROOT,CPE7_REPORT,split='val',device='0')
assert CPE0_REPORT.is_file() and CPE7_REPORT.is_file()
print('CPE0 REPORT:',CPE0_REPORT)
print('CPE7 REPORT:',CPE7_REPORT)


In [ ]:
command=[sys.executable,'-m','pytest','-q','tests/test_circle_cpe.py']
print('STATIC CHECK:',' '.join(command))
subprocess.run(command,cwd=REPO,check=True)


In [ ]:
from coffee_detector.circle_cpe import circle_pair_loss
from coffee_detector.fsce_cpe.loss import cpe_supervised_contrastive_loss
g=torch.Generator().manual_seed(42)
z=torch.randn(200,128,generator=g)
y=torch.arange(200)%21
sup=cpe_supervised_contrastive_loss(z,y,temperature=0.2).item()
cir=circle_pair_loss(z,y,margin=0.25,gamma=256.0).item()
print({'raw_supcon':sup,'weighted_supcon_lambda_0.5':0.5*sup,'raw_circle':cir,'weighted_circle_lambda_0.005':0.005*cir})


In [ ]:
command=[
 sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_circle_cpe_screening',
 '--data-root',str(DATA_ROOT),'--grouped-summary',str(GROUPED_SUMMARY),
 '--d0-checkpoint',str(D0_CHECKPOINT),'--d0ft-report',str(D0FT_REPORT),
 '--cpe0-report',str(CPE0_REPORT),'--cpe7-report',str(CPE7_REPORT),
 '--output-root',str(OUTPUT_ROOT),'--seed','42','--device','0','--authorize-training',
]
print('MENJALANKAN:',' '.join(command),flush=True)
process=subprocess.run(command,cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(process.stdout)
if process.returncode: raise RuntimeError(f'Circle-CPE gagal: {process.returncode}')


In [ ]:
import pandas as pd
from IPython.display import display
SUMMARY=OUTPUT_ROOT/'val_reports/circle_cpe_seed42_screening.json'
result=json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split']=='val'
assert result['test_images_accessed'] is False and result['test_opened'] is False
rows=[]
for name in ('D0FT','CPE0','CPE7'):
    rows.append({'model':name,**result['controls'][name]})
for name,metrics in result['candidate'].items(): rows.append({'model':name,**metrics})
display(pd.DataFrame(rows).style.format({k:'{:.2%}' for k in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}))
for arm in ('CIR0','CIR7'):
    d=result['decisions'][arm]
    print('\n',arm,d['decision'])
    print('matched control:',d['matched_supcon_control'])
    print('delta vs matched CPE:',{k:f'{v*100:+.2f} pp' for k,v in d['delta_vs_matched_cpe'].items()})
    print('delta vs D0FT:',{k:f'{v*100:+.2f} pp' for k,v in d['delta_vs_D0FT'].items()})
    print('criteria:',d['criteria'])
print('\nRETAINED:',result['retained_for_multiseed_confirmation'])
print('NEXT:',result['next_action'])
print('SUMMARY:',SUMMARY)
